## Spark DataFrames Review

In [1]:
!ls

dataframe-operations.ipynb	    spark-dataframes-review.ipynb
missing_data_and_time_series.ipynb  spark_groupby_and_aggregate_functions.ipynb
missing_data.csv		    user_profile.json
sales_data.csv			    user_simple.json
spark-dataframe-basics.ipynb	    WMT.csv


In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Walmart_stock").getOrCreate()

26/06/18 20:30:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
df = spark.read.csv('WMT.csv', header=True, inferSchema=True)

In [39]:
df.columns

['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

In [7]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Adj Close: double (nullable = true)
 |-- Volume: string (nullable = true)



In [8]:
df.head(5)

[Row(Date=datetime.date(2016, 1, 20), Open=61.799999, High=62.330002, Low=60.200001, Close=60.84, Adj Close=53.990601, Volume='17369100'),
 Row(Date=datetime.date(2016, 1, 21), Open=60.98, High=62.790001, Low=60.91, Close=61.880001, Adj Close=54.913509, Volume='12089200'),
 Row(Date=datetime.date(2016, 1, 22), Open=62.439999, High=63.259998, Low=62.130001, Close=62.689999, Adj Close=55.632324, Volume='9197500'),
 Row(Date=datetime.date(2016, 1, 25), Open=62.779999, High=63.82, Low=62.549999, Close=63.450001, Adj Close=56.306763, Volume='12823400'),
 Row(Date=datetime.date(2016, 1, 26), Open=63.360001, High=64.470001, Low=63.259998, Close=64.0, Adj Close=56.794834, Volume='9441200')]

In [10]:
df.describe().show()

26/06/18 20:34:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:>                                                          (0 + 1) / 1]

+-------+-----------------+------------------+------------------+-----------------+------------------+-----------------+
|summary|             Open|              High|               Low|            Close|         Adj Close|           Volume|
+-------+-----------------+------------------+------------------+-----------------+------------------+-----------------+
|  count|             2518|              2518|              2518|             2518|              2518|             2518|
|   mean|96.50830017156473| 97.33101670611606| 95.74480543367737|  96.548617969023| 92.42759966243037|8509442.510925705|
| stddev|23.32285199736492|23.585851735628015|23.015135524430754|23.28824146387443|25.330248130379083|4760469.149170408|
|    min|            60.98|         62.330002|         60.200001|            60.84|         53.990601|         10010500|
|    max|       153.600006|        153.660004|        151.660004|       152.789993|        152.233536|          9999600|
+-------+-----------------+-----

In [11]:
df.describe().printSchema()

root
 |-- summary: string (nullable = true)
 |-- Open: string (nullable = true)
 |-- High: string (nullable = true)
 |-- Low: string (nullable = true)
 |-- Close: string (nullable = true)
 |-- Adj Close: string (nullable = true)
 |-- Volume: string (nullable = true)



In [12]:
from pyspark.sql.functions import format_number

In [13]:
res = df.describe()

In [15]:
df.describe().columns

['summary', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']

In [18]:
res.select(res['summary'],
            format_number(res['Open'].cast('float'), 2).alias('Open'),
            format_number(res['High'].cast('float'), 2).alias('High'),
            format_number(res['Low'].cast('float'), 2).alias('Low'),
            format_number(res['Adj Close'].cast('float'), 2).alias('Close'),
            res['Volume'].try_cast('int').alias('Volume')
            ).show()

+-------+--------+--------+--------+--------+--------+
|summary|    Open|    High|     Low|   Close|  Volume|
+-------+--------+--------+--------+--------+--------+
|  count|2,518.00|2,518.00|2,518.00|2,518.00|    2518|
|   mean|   96.51|   97.33|   95.74|   92.43|    NULL|
| stddev|   23.32|   23.59|   23.02|   25.33|    NULL|
|    min|   60.98|   62.33|   60.20|   53.99|10010500|
|    max|  153.60|  153.66|  151.66|  152.23| 9999600|
+-------+--------+--------+--------+--------+--------+



In [19]:
# High vs Volume
df2 = df.withColumn('HV Ratio', df['High']/df['Volume'])

In [20]:
df2.show()

+----------+---------+---------+---------+---------+---------+--------+--------------------+
|      Date|     Open|     High|      Low|    Close|Adj Close|  Volume|            HV Ratio|
+----------+---------+---------+---------+---------+---------+--------+--------------------+
|2016-01-20|61.799999|62.330002|60.200001|    60.84|53.990601|17369100|3.588556804900656...|
|2016-01-21|    60.98|62.790001|    60.91|61.880001|54.913509|12089200|5.193892151672566...|
|2016-01-22|62.439999|63.259998|62.130001|62.689999|55.632324| 9197500|6.877955748844795E-6|
|2016-01-25|62.779999|    63.82|62.549999|63.450001|56.306763|12823400|  4.9768392158086E-6|
|2016-01-26|63.360001|64.470001|63.259998|     64.0|56.794834| 9441200|6.828581218489175E-6|
|2016-01-27|64.099998|    65.18|63.889999|63.950001|56.750477|10214300|6.381249816433823...|
|2016-01-28|64.029999|64.510002|    63.43|64.220001| 56.99007|11278300|5.719833840206414E-6|
|2016-01-29|    64.75|66.529999|64.739998|66.360001|58.889149|16439100

In [21]:
df2.select('HV Ratio').show()

+--------------------+
|            HV Ratio|
+--------------------+
|3.588556804900656...|
|5.193892151672566...|
|6.877955748844795E-6|
|  4.9768392158086E-6|
|6.828581218489175E-6|
|6.381249816433823...|
|5.719833840206414E-6|
|4.047058476437275E-6|
|4.612177833301649E-6|
|  4.9934119933166E-6|
|5.480853551593101E-6|
|5.185687580843736E-6|
|4.756806184622971E-6|
|3.237143118841474E-6|
|4.535458941157187E-6|
|6.858372488232931E-6|
|5.878409361116325...|
|6.833066886700015E-6|
|5.880023150389508E-6|
|5.360232483281965...|
+--------------------+
only showing top 20 rows


In [22]:
df.orderBy(df['High'].desc()).head(1)[0][0]

datetime.date(2020, 12, 1)

In [23]:
from pyspark.sql.functions import mean

df.select(mean('Close')).show()

+---------------+
|     avg(Close)|
+---------------+
|96.548617969023|
+---------------+



In [24]:
from pyspark.sql.functions import max, min

In [25]:
df.select(max('Volume'), min('Volume')).show()

+-----------+-----------+
|max(Volume)|min(Volume)|
+-----------+-----------+
|    9999600|   10010500|
+-----------+-----------+



In [28]:
df.filter('Close<62').count()

4

In [29]:
from pyspark.sql.functions import count

In [30]:
res = df.filter('Close<62')
res.select(count('Close')).show()

+------------+
|count(Close)|
+------------+
|           4|
+------------+



In [33]:
(df.filter('High>80').count() * 1.0/df.count())*100

68.06989674344717

In [42]:
from pyspark.sql.functions import corr, col

In [46]:
df.select(corr('High', col('Volume').try_cast('int'))).show()

+-----------------------------------+
|corr(High, TRY_CAST(Volume AS INT))|
+-----------------------------------+
|               -0.07964751035332947|
+-----------------------------------+



In [47]:
from pyspark.sql.functions import year

year_df = df.withColumn('Year', year(df['Date']))

In [48]:
max_df = year_df.groupBy('Year').max()

In [49]:
max_df.select('Year', 'max(High)').show()

+----+----------+
|Year| max(High)|
+----+----------+
|2018|109.980003|
|2019|125.379997|
|2020|153.660004|
|2016| 75.190002|
|2017|100.129997|
|2021|149.929993|
+----+----------+



In [50]:
max_df.show()

+----+----------+----------+----------+----------+--------------+---------+
|Year| max(Open)| max(High)|  max(Low)|max(Close)|max(Adj Close)|max(Year)|
+----+----------+----------+----------+----------+--------------+---------+
|2018|109.139999|109.980003|107.989998|109.550003|    103.163734|     2018|
|2019|124.599998|125.379997|120.699997|121.279999|    119.288666|     2019|
|2020|153.600006|153.660004|151.660004|152.789993|    152.233536|     2020|
|2016|      74.5| 75.190002| 73.629997| 74.300003|     67.367744|     2016|
|2017| 99.910004|100.129997| 99.120003| 99.620003|     93.605438|     2017|
|2021|     149.0|149.929993|148.320007|148.970001|    148.970001|     2021|
+----+----------+----------+----------+----------+--------------+---------+



In [51]:
from pyspark.sql.functions import month

In [52]:
month_df = df.withColumn('Month', month('Date'))
monthavgs = month_df.select('Month', 'Close').groupBy('Month').mean()
monthavgs.select('Month', 'avg(Close)').orderBy('Month').show()

+-----+------------------+
|Month|        avg(Close)|
+-----+------------------+
|    1| 98.94980368627448|
|    2|  89.1636457083333|
|    3| 87.44880724770643|
|    4| 91.55893247572817|
|    5| 90.54859816822429|
|    6| 92.23028014018693|
|    7| 96.65647596190473|
|    8| 96.97705391071433|
|    9|100.69396066336634|
|   10|102.74810810810816|
|   11|105.59009729126213|
|   12|106.02932022330101|
+-----+------------------+

